# SR-EV Structure Visualization

Exports SR-EV structures for **OVITO**, and previews them inside the notebook. Everything here goes through `hmt_v3.srev.visualize`, which offers three views of the same structure fed by one shared assembly step (`build_particle_table`), so they all agree on species, colors, radii and units:

| | what it is | notes |
|---|---|---|
| `export_for_ovito` | extended XYZ for OVITO | large scenes, rendering, OVITO's analysis modifiers |
| `plot_structure_3d` | static matplotlib, **inline** | quick look / figure panel; works headless |
| `view_napari` | interactive 3D from Python | best for exploring; needs an OpenGL context |

The three input shapes are each covered below: **raw** SR-EV, **het/eu labeled**, and **labeled + fluorophores** (where the two dye species get distinct types and colors).

**Units:** positions are written in nanometres, so distances measured in OVITO read directly in nm. Opening a raw `config-N.txt` in OVITO instead works (it is already plain XYZ) but gives the generator's native Umin units, 10x smaller, with no compartment or coordination information.

In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.spatial import cKDTree

from hmt_v3 import srev

CONFIG_PATH = Path("hmt_v1/Simulation/config-1.txt")  # any config-N.txt(.gz)
OUT = Path("ovito_export")

## Load the config

`resolve_srev_run_params` reads a sibling `copy-parameters.txt` or an `SRRW-*` folder name when there is one; it returns whatever it can find without raising, so fall back to `length_unit_to_nm=10` (Umin) when neither is available.

In [ ]:
params = srev.io.resolve_srev_run_params(CONFIG_PATH)
length_unit_to_nm = params.get("length_unit_to_nm", 10.0)

raw, meta = srev.io.parse_config_txt(CONFIG_PATH, length_unit_to_nm=length_unit_to_nm)
print(f"{len(raw)} monomers, length_unit_to_nm={length_unit_to_nm}")
print(f"extent: x {raw['x [nm]'].min():.0f}..{raw['x [nm]'].max():.0f} nm, "
      f"y {raw['y [nm]'].min():.0f}..{raw['y [nm]'].max():.0f} nm, "
      f"z {raw['z [nm]'].min():.0f}..{raw['z [nm]'].max():.0f} nm")

## 1. Raw SR-EV

Coordinates only -- no compartments, no coordination number. Everything is one untyped `MONO` population.

In [ ]:
result_raw = srev.visualize.export_for_ovito(OUT, monomers_df=raw, basename="raw")
result_raw

## 2. Het/eu labeled

`classify_compartments` adds a coordination number per monomer (neighbours within 11.5 nm, range 0-12) and splits them into heterochromatin / euchromatin. Monomers now export as two types, `HET` and `EUC` -- **green** and **red** respectively, matching the convention already used for the localization track in `hmt_v3/visualize.py` (H3K27me3 marks heterochromatin, H3K27ac marks euchromatin).

Three variants are written below, because the choice interacts with how OVITO decides what to display:

| file | `color_by` | `include_color` | what OVITO shows |
|---|---|---|---|
| `labeled.xyz` | `"species"` (default) | `True` | green / red, baked in |
| `labeled_cn.xyz` | `"coordination_number"` | `True` | CN shading, baked in |
| `labeled_bytype.xyz` | — | `False` | OVITO's own **particle type** colors, which you set from its type list |

**The catch worth knowing:** OVITO gives an explicit `Color` property priority over its per-type colors. So with `include_color=True` the file's colors win, and editing colors in OVITO's type list has no visible effect. Pass `include_color=False` when you want to drive the colors yourself. A **Color coding** modifier overrides either, since it runs later in OVITO's pipeline.

In [ ]:
labeled = srev.compartments.classify_compartments(raw)
print(labeled["compartment"].value_counts().to_dict())
print(f"CN range: {labeled['coordination_number'].min()}-{labeled['coordination_number'].max()}")

# green HET / red EUC baked in (color_by="species" is the default)
result_labeled = srev.visualize.export_for_ovito(
    OUT, monomers_df=labeled, basename="labeled",
)

# same structure, CN shading baked in instead
result_labeled_cn = srev.visualize.export_for_ovito(
    OUT, monomers_df=labeled, basename="labeled_cn", color_by="coordination_number",
)

# no Color property at all -- OVITO colors by particle type, and its type list drives it
result_labeled_bytype = srev.visualize.export_for_ovito(
    OUT, monomers_df=labeled, basename="labeled_bytype", include_color=False,
)

for r in (result_labeled, result_labeled_cn, result_labeled_bytype):
    print(f"{Path(r['structure']).name:22s} {r['n_particles']:>7d} particles  species={r['species']}")

## 3. Labeled + fluorophores

The **whole structure** is labeled -- no cropping. All 186,742 monomers go through antibody and dye placement in about 2 s, so there is nothing to gain by cutting it down, and a cropped region can only misrepresent the morphology.

> **If you ever do need to crop** (a much larger config, say), use a **box**, not k-nearest-neighbours. `tree.query(centroid, k=N)` returns the N monomers *closest* to a point, which is by construction a filled ball: it keeps 100% of the config inside some radius and discards everything outside, so the crop boundary itself becomes a smooth spherical surface and the void structure disappears. Measured voxel occupancy (20 nm voxels): a KNN ball reads **38.7%** filled versus **13.7%** for the real config -- it looks like a solid sphere of chromatin because that is what it is. A box keeps domains, voids and the corrugated surface intact.
>
> Either way, classify compartments on the **full** cloud first: coordination number counts real neighbours, so classifying after cropping undercounts at the boundary and biases those monomers toward euchromatin.

### Single labeling run -- dye channel resolved automatically

With one labeling run over a compartment-classified patch, each dye's channel is recovered by walking `fluorophore -> secondary -> primary -> monomer -> compartment`. That is what splits the dyes into the `DYE_HET` and `DYE_EUC` types, drawn in bright spring green and bright pink-red against the muted green/red monomers.

**Only the monomers and dyes are exported.** The antibodies still have to be *passed* -- without them the FK walk above cannot tell `DYE_HET` from `DYE_EUC` -- but `layers=("monomers", "fluorophores", "domains")` keeps them out of the picture.

**Every point draws at the same radius** (r° = 4.9 nm). This view is about *where* things are, not how big they are: at true relative scale a dye is a ~1 nm organic molecule against an ~11 nm nucleosome and is effectively invisible next to it. For a volume-accurate picture instead, pass `label_radius_nm=5.0` (IgG hydrodynamic radius) and `dye_radius_nm=0.7`. None of it affects the simulation -- placement is governed by the linker lengths and `exclusion_radius_nm` in `LabelingConfig`.

In [ ]:
primaries, secondaries, fluorophores = srev.labeling.label_monomers(
    labeled, srev.labeling.LabelingConfig(), rng=np.random.default_rng(0),
)
print(f"{len(primaries)} primaries, {len(secondaries)} secondaries, {len(fluorophores)} fluorophores")
print(f"dyes per secondary: {len(fluorophores)/len(secondaries):.2f} (requested Poisson lambda=3.0)")

channels = srev.visualize.resolve_fluorophore_channel(fluorophores, secondaries, primaries, labeled)
print(f"resolved dye channels: {pd.Series(channels).value_counts().to_dict()}")

# packing domains for context -- these come from a single projected z-slab, so
# they carry x/y only and need the slab centre passed back in as their z
z_center = float(labeled["z [nm]"].median())
domains = srev.domains.identify_packing_domains(labeled, z_center_nm=z_center)
print(f"{len(domains)} packing domains, radius {domains['radius_nm'].min():.0f}"
      f"-{domains['radius_nm'].max():.0f} nm")

# The antibodies are still PASSED -- the dye-channel FK walk needs them to tell
# DYE_HET from DYE_EUC -- but layers= keeps them out of the exported picture.
result_dyes = srev.visualize.export_for_ovito(
    OUT, monomers_df=labeled, primaries_df=primaries, secondaries_df=secondaries,
    fluorophores_df=fluorophores, domains_df=domains, domain_z_nm=z_center,
    layers=("monomers", "fluorophores", "domains"),
    basename="labeled_dyes",
)
result_dyes

### Two channels from separate runs

A two-channel acquisition is two independent `pipeline.run_pipeline` calls (one per compartment), so the channel is already known and there is nothing to resolve -- pass it explicitly with `channel=` and concatenate the tables. Build the monomers into one of them only, so they are not duplicated.

In [ ]:
het_monomers = labeled[labeled["compartment"] == "heterochromatin"].reset_index(drop=True).copy()
eu_monomers = labeled[labeled["compartment"] == "euchromatin"].reset_index(drop=True).copy()
het_monomers["monomer_id"] = np.arange(len(het_monomers), dtype=np.int64)
eu_monomers["monomer_id"] = np.arange(len(eu_monomers), dtype=np.int64)

_, _, fluor_het = srev.labeling.label_monomers(
    het_monomers, srev.labeling.LabelingConfig(), rng=np.random.default_rng(1))
_, _, fluor_eu = srev.labeling.label_monomers(
    eu_monomers, srev.labeling.LabelingConfig(), rng=np.random.default_rng(2))
print(f"monomers: het={len(het_monomers)}, eu={len(eu_monomers)}")
print(f"dyes:     het={len(fluor_het)}, eu={len(fluor_eu)}")
print("note euchromatin can yield MORE dyes despite having fewer monomers -- it is less "
      "crowded, so fewer placements are lost to steric rejection")

particles = pd.concat([
    srev.visualize.build_particle_table(monomers_df=labeled, fluorophores_df=fluor_het,
                                        channel="heterochromatin"),
    srev.visualize.build_particle_table(fluorophores_df=fluor_eu,
                                        channel="euchromatin"),
], ignore_index=True)

result_two = srev.visualize.export_for_ovito(OUT, particles=particles, basename="two_channel")
result_two

## 4. sSMLM image -- the simulated camera movie

Everything above exports the *structure*. This exports the **data**: `pipeline.run_pipeline` takes the labeled monomers through blinking photophysics, astigmatic PSF rendering and the EMCCD noise chain, and writes a `stack.tif` per channel -- the raw movie ThunderSTORM consumes -- alongside the ground-truth tables to score its output against.

Emissions are genuinely separated in time: `simulate_blinking` steps a 3-state Markov chain (Off/On/Bleached) frame by frame with persistent per-emitter state, so each dye switches independently and stays on for a correlated run of frames, and only frames where it was actually On get a PSF drawn.

**But the activation RATE has to be set for sparsity, and the default is wrong for this structure.** SMLM only resolves single molecules if at most ~1 emitter per µm² is on per frame. This channel packs ~37,000 dyes into ~1.7 µm², so `PhotophysicsConfig`'s default `k_on_per_ms=1e-5` lights ~20 at once and every PSF overlaps into one blob:

| `k_on_per_ms` | frames | blinks | on/frame | density | usable? |
|---|---|---|---|---|---|
| 1e-5 (default) | 2,000 | 7,385 | 19.6 | 11.6 /µm² | no -- fully merged |
| 1e-6 | 5,000 | 1,887 | 2.0 | 1.21 /µm² | marginal |
| **5e-7** | **20,000** | **3,714** | **0.99** | **0.58 /µm²** | **yes** |
| 2e-7 | 20,000 | 1,441 | 0.37 | 0.22 /µm² | yes, but sparse |

That is the real trade-off: sparser frames need proportionally more of them to collect enough localizations. 20,000 frames is squarely in the normal experimental range and renders in a few seconds. The cell below reports the achieved density so you can check it rather than assume it.

In [ ]:
import tifffile

SSMLM_OUT = Path("ssmlm_export")
N_FRAMES = 20000
K_ON_PER_MS = 5e-7   # see the table above -- NOT the PhotophysicsConfig default


def active_per_frame(blink_events, n_frames):
    """How many emitters are On in each frame -- the sparsity SMLM depends on."""
    on = np.zeros(n_frames, dtype=int)
    for start, end in zip(blink_events["frame_start"], blink_events["frame_end"]):
        on[start:end + 1] += 1
    return on


runs = {}
for channel, seed in (("heterochromatin", 1), ("euchromatin", 2)):
    config = srev.pipeline.SrEvSimConfig(
        photophysics=srev.photophysics.PhotophysicsConfig(
            k_on_per_ms=K_ON_PER_MS, n_frames=N_FRAMES),
        camera=srev.render.CameraConfig(pixel_size_nm=100.0),
        compartment_filter=channel,
        seed=seed,
    )
    run = srev.pipeline.run_pipeline(labeled, config, SSMLM_OUT / channel,
                                     source_path=CONFIG_PATH, input_meta=meta)
    runs[channel] = run

    fl = run["fluorophores"]
    area_um2 = ((fl["x [nm]"].max() - fl["x [nm]"].min())
                * (fl["y [nm]"].max() - fl["y [nm]"].min())) / 1e6
    on = active_per_frame(run["blink_events"], N_FRAMES)
    tif_mb = Path(run["written"]["tiff"]).stat().st_size / 1e6

    print(f"[{channel}]")
    print(f"   {len(fl)} dyes -> {len(run['blink_events'])} blinks over {N_FRAMES} frames")
    print(f"   frame {run['frame_shape']} px, stack.tif {tif_mb:.1f} MB")
    print(f"   active emitters/frame: mean {on.mean():.2f}, max {on.max()}"
          f"  ->  {on.mean()/area_um2:.2f} per um^2   (SMLM wants <~1)")
    print(f"   mean on-time {run['blink_events']['n_frames_on'].mean():.2f} frames, "
          f"end reasons {run['blink_events']['end_reason'].value_counts().to_dict()}")

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 9))

for row, channel in enumerate(("heterochromatin", "euchromatin")):
    run = runs[channel]
    stack = tifffile.imread(run["written"]["tiff"])
    gt = run["blink_events"].merge(
        run["fluorophores"][["fluorophore_id", "x [nm]", "y [nm]"]],
        on="fluorophore_id", how="left")
    on = active_per_frame(run["blink_events"], stack.shape[0])
    busiest = int(np.argmax(on))
    px = 100.0

    # 1. a single raw frame -- individual PSFs, which is what a fitter has to work with
    axes[row, 0].imshow(stack[busiest], cmap="gray")
    axes[row, 0].set_title(f"{channel}\nraw frame {busiest} -- {on[busiest]} emitters on")

    # 2. summed over the movie: the diffraction-limited image SMLM is trying to beat
    axes[row, 1].imshow(stack.sum(axis=0), cmap="inferno")
    axes[row, 1].set_title(f"sum of {stack.shape[0]} frames\n(diffraction-limited equivalent)")

    # 3. ground-truth localizations at 10 nm pixels: the perfect reconstruction that a
    #    real ThunderSTORM run gets scored against
    edges_y = np.arange(0, stack.shape[1] * px + 10.0, 10.0)
    edges_x = np.arange(0, stack.shape[2] * px + 10.0, 10.0)
    hist, _, _ = np.histogram2d(gt["y [nm]"], gt["x [nm]"], bins=[edges_y, edges_x])
    vmax = np.percentile(hist[hist > 0], 99) if (hist > 0).any() else 1
    axes[row, 2].imshow(hist, cmap="inferno", vmax=vmax)
    axes[row, 2].set_title(f"ground-truth localizations\n{len(gt)} blinks at 10 nm px")

    for ax in axes[row]:
        ax.set_xticks([])
        ax.set_yticks([])

fig.tight_layout()
plt.show()

## Inline preview

`plot_structure_3d` renders in the notebook with no viewer install and works headless. It thins large clouds automatically (matplotlib's 3D scatter degrades long before napari does), so treat it as a sanity check rather than the real view.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 7), subplot_kw={"projection": "3d"})

srev.visualize.plot_structure_3d(
    monomers_df=labeled, color_by="compartment", ax=axes[0],
    title="Full structure -- green heterochromatin / red euchromatin",
)

# A thin z-slab for the second panel: 75k dyes over 187k monomers is an opaque wall
# from any viewing angle. A slab keeps both populations legible and shows where the
# dyes sit relative to the structure. Dyes take their channel color regardless of
# color_by, so they stay distinct from the monomers underneath.
SLAB_HALF_NM = 40.0
z_mid = labeled["z [nm]"].median()
mono_slab = labeled[(labeled["z [nm]"] - z_mid).abs() <= SLAB_HALF_NM]
fluor_slab = fluorophores[(fluorophores["z [nm]"] - z_mid).abs() <= SLAB_HALF_NM]
print(f"slab: {len(mono_slab)} monomers, {len(fluor_slab)} dyes")

srev.visualize.plot_structure_3d(
    particles=srev.visualize.build_particle_table(
        monomers_df=mono_slab, fluorophores_df=fluor_slab),
    color_by="compartment", ax=axes[1], point_size=4.0,
    title=f"{2*SLAB_HALF_NM:.0f} nm slab -- dyes in context",
)
plt.tight_layout()
plt.show()

## Interactive view

`view_napari` opens a real 3D viewer driven from Python, with each species on its own toggleable layer. It needs an OpenGL context, so it will **not** run under `nbconvert --execute` -- uncomment and run it in a normal Jupyter session.

In [ ]:
# Opens a real viewer window when run interactively. Guarded because napari needs
# an OpenGL context, which `nbconvert --execute` and other headless runs do not have.
try:
    srev.visualize.view_napari(particles=particles)
except Exception as exc:
    print(f"napari viewer unavailable here ({type(exc).__name__}) -- "
          f"run this notebook in an interactive Jupyter session to open it")

# or straight from the tables, antibodies used for channel resolution but not shown:
# srev.visualize.view_napari(
#     monomers_df=labeled, primaries_df=primaries, secondaries_df=secondaries,
#     fluorophores_df=fluorophores, domains_df=domains, domain_z_nm=z_center,
#     layers=("monomers", "fluorophores", "domains"),
# )

for layer in srev.visualize.structure_layers(particles):
    print(f"{layer['name']:34s} n={len(layer['data']):6d}  "
          f"diameter {layer['size'].min():.1f}-{layer['size'].max():.1f} nm")

## What was written

In [ ]:
print("structure exports (OVITO):")
for f in sorted(OUT.glob("*.xyz")):
    n_particles = int(f.read_text().splitlines()[0])
    print(f"  {f.name:20s} {f.stat().st_size / 1e6:6.1f} MB  {n_particles:>7d} particles")

print("\nsSMLM runs (ThunderSTORM input + ground truth):")
for d in sorted(p for p in SSMLM_OUT.iterdir() if p.is_dir()):
    print(f"  {d.name}/")
    for f in sorted(d.iterdir()):
        print(f"    {f.name:34s} {f.stat().st_size / 1e6:7.2f} MB")

## Opening these in OVITO

File > Load File on any `.xyz` above.

**If OVITO is showing colors you did not ask for**, it is the baked `Color` property: OVITO gives an explicit `Color` priority over its own per-type colors, so whatever `color_by` was used at export wins and the type list's colors are masked. Three ways out:

- open `labeled_bytype.xyz`, exported with `include_color=False` -- no `Color` property, so OVITO colors by particle type and its **type list** drives everything
- re-export with a different `color_by`
- add a **Color coding** modifier, which runs later in OVITO's pipeline and overrides the baked colors either way

Worth trying:

- **Particle types panel** -- `HET`/`EUC` (green/red), and `DYE_HET`/`DYE_EUC` in the dye exports. Recolor, resize or hide each independently (colors only take visible effect on a file exported with `include_color=False`).
- **Color coding** modifier on `Coordination` (range 0-12), or on `compartment` (0 none / 1 heterochromatin / 2 euchromatin) and `layer` (0 monomers / 1 primaries / 2 secondaries / 3 fluorophores / 4 domains).
- **Coordination analysis** modifier with cutoff `11.5` -- OVITO computes the same quantity `srev.compartments.compute_coordination_number` does, so color-coding OVITO's result and comparing against the `Coordination` already in the file independently checks that implementation.
- **Slice** modifier to cut a 100 nm slab, matching what `domains.identify_packing_domains` operates on.

`ovito_export/` is regenerated every time this notebook runs, and is in `.gitignore`.